In [1]:
from dataclasses import dataclass, field
from typing import List, Dict, Optional
import pandas as pd
import numpy as np
import rasterio
import geopandas as gpd
from datetime import datetime
import matplotlib.pyplot as plt
import sys
sys.path.append('/Users/michaelfoley/Library/CloudStorage/GoogleDrive-mfoley@g.harvard.edu/My Drive/Subnational_Yield_Database/scripts/global/boundaries_processing')
import relationships
import seaborn as sns

In [2]:
from datetime import datetime

current_date = datetime.now().strftime('%m%d%y')
print(current_date)

120425


# Assess what states and districts we have at the start of 2016

In [3]:
india_relationship = pd.read_csv('../shapefiles/relationshiptable_IN.csv')
india_changes = pd.read_excel('../shapefiles/IN_Relationships_units.xlsx')

In [4]:
india_relationship.head()

,label,from_unit,to_unit,relationship_type,category,from_unit_name,to_unit_name,fnid_from_unit,fnid_to_unit,can_aggregate_to_from,can_aggregate_from_to
0,Dhanbad redistribute Bokaro,127389,128198,redistribute,temporal,"Dhanbad, Bihar, India","Bokaro, Bihar, India",IN1996A20513,IN1998A20518,False,False
1,Andaman Islands successor Andaman Islands,127322,127990,successor,temporal,"Andaman Islands, Andaman and Nicobar Islands, ...","Andaman Islands, Andaman and Nicobar Islands, ...",IN1996A20101,IN1998A20101,True,True
2,Nicobar Islands successor Nicobar Islands,127323,127991,successor,temporal,"Nicobar Islands, Andaman and Nicobar Islands, ...","Nicobar Islands, Andaman and Nicobar Islands, ...",IN1996A20102,IN1998A20102,True,True
3,North and Middle Andaman successor North and M...,127324,127992,successor,temporal,"North and Middle Andaman, Andaman and Nicobar ...","North and Middle Andaman, Andaman and Nicobar ...",IN1996A20103,IN1998A20103,True,True
4,South Andaman successor South Andaman,127325,127993,successor,temporal,"South Andaman, Andaman and Nicobar Islands, India","South Andaman, Andaman and Nicobar Islands, India",IN1996A20104,IN1998A20104,True,True


In [5]:
india_changes.head()

,relationship_type,from_unit_FNID,from_unit_Name,to_unit_FNID,to_unit_Name,State,U_ID
0,Split,IN1982A20405,"Kamrup (AS), Assam",IN1983A20412,"Barpeta (AS), Assam",AS,0405
1,Split,IN1982A20401,"Cachar (AS), Assam",IN1983A20413,"Cachar (AS), Assam",AS,0401
2,Split,IN1982A20401,"Cachar (AS), Assam",IN1983A20414,"Karimganj (AS), Assam",AS,0401
3,Split,IN1982A20402,"Darrang (AS), Assam",IN1983A20415,"Darrang (AS), Assam",AS,0402
4,Split,IN1982A20402,"Darrang (AS), Assam",IN1983A20416,"Sonitpur (AS), Assam",AS,0402


In [9]:
india_relationship

,label,from_unit,to_unit,relationship_type,category,from_unit_name,to_unit_name,fnid_from_unit,fnid_to_unit,can_aggregate_to_from,can_aggregate_from_to
0,Dhanbad redistribute Bokaro,127389,128198,redistribute,temporal,"Dhanbad, Bihar, India","Bokaro, Bihar, India",IN1996A20513,IN1998A20518,False,False
1,Andaman Islands successor Andaman Islands,127322,127990,successor,temporal,"Andaman Islands, Andaman and Nicobar Islands, ...","Andaman Islands, Andaman and Nicobar Islands, ...",IN1996A20101,IN1998A20101,True,True
2,Nicobar Islands successor Nicobar Islands,127323,127991,successor,temporal,"Nicobar Islands, Andaman and Nicobar Islands, ...","Nicobar Islands, Andaman and Nicobar Islands, ...",IN1996A20102,IN1998A20102,True,True
3,North and Middle Andaman successor North and M...,127324,127992,successor,temporal,"North and Middle Andaman, Andaman and Nicobar ...","North and Middle Andaman, Andaman and Nicobar ...",IN1996A20103,IN1998A20103,True,True
4,South Andaman successor South Andaman,127325,127993,successor,temporal,"South Andaman, Andaman and Nicobar Islands, India","South Andaman, Andaman and Nicobar Islands, India",IN1996A20104,IN1998A20104,True,True
...,...,...,...,...,...,...,...,...,...,...,...
9940,New Delhi successor New Delhi,127434,128113,successor,temporal,"New Delhi, Delhi, India","New Delhi, Delhi, India",IN1996A20807,IN1998A20807,True,True
9941,North successor North,127435,128114,successor,temporal,"North, Delhi, India","North, Delhi, India",IN1996A20808,IN1998A20808,True,True
9942,Nalgonda split Test 1,0,0,split,temporal,"Nalgonda, IN","Test 1, IN",IN2015A23707,IN2016A201,True,False
9943,Nalgonda split Test 2,0,0,split,temporal,"Nalgonda, IN","Test 2, IN",IN2015A23707,IN2016A201,True,False


In [10]:
# count rows where 'to_unit_FNID' contains '2016'
count_2016 = india_relationship['fnid_to_unit'].astype(str).str.contains('2016', na=False).sum()
print(count_2016)

3


In [19]:
# We need to identify all rows where 2016 falls between the from and to years
india_relationship['from_year'] = india_relationship['fnid_from_unit'].fillna(0).str.split('IN').str[1].str[:4].astype(int)
india_relationship['to_year'] = india_relationship['fnid_to_unit'].fillna(0).str.split('IN').str[1].str[:4].astype(int)

In [20]:
which_2016 = india_relationship[(india_relationship['from_year'] <= 2016) & (india_relationship['to_year'] >= 2016)]

In [25]:
india_relationship

,label,from_unit,to_unit,relationship_type,category,from_unit_name,to_unit_name,fnid_from_unit,fnid_to_unit,can_aggregate_to_from,can_aggregate_from_to,from_year,to_year
0,Dhanbad redistribute Bokaro,127389,128198,redistribute,temporal,"Dhanbad, Bihar, India","Bokaro, Bihar, India",IN1996A20513,IN1998A20518,False,False,1996,1998
1,Andaman Islands successor Andaman Islands,127322,127990,successor,temporal,"Andaman Islands, Andaman and Nicobar Islands, ...","Andaman Islands, Andaman and Nicobar Islands, ...",IN1996A20101,IN1998A20101,True,True,1996,1998
2,Nicobar Islands successor Nicobar Islands,127323,127991,successor,temporal,"Nicobar Islands, Andaman and Nicobar Islands, ...","Nicobar Islands, Andaman and Nicobar Islands, ...",IN1996A20102,IN1998A20102,True,True,1996,1998
3,North and Middle Andaman successor North and M...,127324,127992,successor,temporal,"North and Middle Andaman, Andaman and Nicobar ...","North and Middle Andaman, Andaman and Nicobar ...",IN1996A20103,IN1998A20103,True,True,1996,1998
4,South Andaman successor South Andaman,127325,127993,successor,temporal,"South Andaman, Andaman and Nicobar Islands, India","South Andaman, Andaman and Nicobar Islands, India",IN1996A20104,IN1998A20104,True,True,1996,1998
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9940,New Delhi successor New Delhi,127434,128113,successor,temporal,"New Delhi, Delhi, India","New Delhi, Delhi, India",IN1996A20807,IN1998A20807,True,True,1996,1998
9941,North successor North,127435,128114,successor,temporal,"North, Delhi, India","North, Delhi, India",IN1996A20808,IN1998A20808,True,True,1996,1998
9942,Nalgonda split Test 1,0,0,split,temporal,"Nalgonda, IN","Test 1, IN",IN2015A23707,IN2016A201,True,False,2015,2016
9943,Nalgonda split Test 2,0,0,split,temporal,"Nalgonda, IN","Test 2, IN",IN2015A23707,IN2016A201,True,False,2015,2016


# Load in the aggregated data file to see what districts are present (prior to hybridization) in 2016

In [26]:
from datetime import datetime

current_date = datetime.now().strftime('%m%d%y')
print(current_date)

120425


In [27]:
unmerged_stats = pd.read_csv(f"../../data/ag_stats_final_{current_date}.csv")
unmerged_stats.columns

Index(['Data Source Organization', 'Data Source Document', 'Publication Name',
       'Survey Type', 'Country', 'Zone', 'FNID', 'Admin 1', 'Admin 2',
       'Admin 3', 'Admin 4', 'Year', 'Start Period', 'Season', 'End Period',
       'Crop', 'Dominant Production System', 'Area Planted: ha',
       'Area Harvested: ha', 'Yield: MT/ha (Reported)',
       'Yield: MT/ha (Calculated)', 'Quantity Produced: MT',
       'Contributions by', 'Source crop'],
      dtype='object')